## GroupBy, Join, and Shuffling — how they actually work

Spark splits your data across many workers (partitions), and each worker only sees its own slice. That's fine for row-by-row work, but `GROUP BY` and `JOIN` both need to compare rows that might live on *different* workers. That's where shuffling comes in.

### What is shuffling?

Shuffling is the process of moving data between workers so that related rows end up together on the same worker. It involves:
1. Each worker writes its data to disk, organized by some key (e.g. `zone`, `hour`).
2. Data is sent across the network to the worker that "owns" that key.
3. The receiving worker reads it back in and continues processing.

This is slow compared to everything else Spark does, because it touches disk and network instead of just memory. It's the main thing to watch out for when writing Spark jobs.

### GroupBy — how it happens

When we run something like:
```sql
GROUP BY hour, zone
```
Spark can't just add up numbers randomly across workers — every row for a given `(hour, zone)` needs to end up together so it can be summed/counted correctly. So Spark:
1. **Pre-combines locally first.** Each worker sums/counts the rows it already has for each key, right where the data sits (this is like doing subtotals before mailing them in).
2. **Shuffles the subtotals.** Those smaller, pre-combined results are what actually get moved across the network — not the raw rows — which keeps the shuffle relatively light.
3. **Combines the subtotals into a final answer** on whichever worker now owns each key.

So a `GROUP BY` always causes one shuffle, but Spark tries to shrink how much data crosses the network by aggregating first.

### Join — how it happens

When we do:
```python
df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')
```
Spark needs matching `(hour, zone)` rows from both DataFrames to sit on the same worker so it can pair them up. There are two ways this can happen:

- **Shuffle join (the default for two big tables):** Both DataFrames get shuffled so that matching keys land on the same worker, then Spark matches rows locally. This is the same expensive shuffle-across-the-network step as `GROUP BY`, but now it happens to *two* datasets instead of one.
- **Broadcast join (used when one side is small):** Instead of shuffling both sides, Spark just copies the small DataFrame in full to every worker. Then each worker can join its local slice of the big DataFrame against the full copy of the small one, with no shuffle needed for the big table at all. This is why joining the (tiny) `zones` lookup table later in the notebook is much cheaper than joining `green` and `yellow` revenue together.

### Why this matters for this notebook

This notebook does three shuffle-triggering steps back to back: group green trips, group yellow trips, then join the two, then join again with the zones table. Each one is a separate shuffle stage where data gets written out, sent over the network, and read back in. That's the real cost of the pipeline — not the SQL itself, but the data movement needed to get matching keys together. Reducing the number of joins/groupBys, pre-aggregating before joining, and letting Spark broadcast small lookup tables (like `zones`) are the main ways to keep that cost down.


In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/harsh/Data-engineering-zoomcamp/batch_processing_spark/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/13 14:55:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

26/08/13 14:56:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

In [4]:
df_green.createOrReplaceTempView ('green')

In [5]:
# performing group by to select data from df_green;
# The new table is grouped based on the hour, zone
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [6]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [7]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.createOrReplaceTempView('yellow')

26/08/13 14:59:47 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/yellow/*/*.
java.io.FileNotFoundException: File data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [8]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [9]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [10]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [11]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [12]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [13]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [14]:
df_join = spark.read.parquet('data/report/revenue/total')

In [15]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [16]:
 df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [17]:
df.write.parquet('zones',mode='overwrite')

In [18]:
df_zones = spark.read.parquet('zones/')

In [19]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [22]:
df_result.printSchema()

root
 |-- hour: timestamp (nullable = true)
 |-- zone: integer (nullable = true)
 |-- green_amount: double (nullable = true)
 |-- green_number_records: long (nullable = true)
 |-- yellow_amount: double (nullable = true)
 |-- yellow_number_records: long (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [20]:
# for this report 'LocationID', 'zone' are not needed so droping them
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

In [21]:
df_result.printSchema()

root
 |-- hour: timestamp (nullable = true)
 |-- zone: integer (nullable = true)
 |-- green_amount: double (nullable = true)
 |-- green_number_records: long (nullable = true)
 |-- yellow_amount: double (nullable = true)
 |-- yellow_number_records: long (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [23]:
spark.stop()